# 예외 항목 데이터 전처리 및 rag 연결

- 기존 임베딩한 모델 로드 및 기존 백터 저장소 불러오기
- 이후 text_spitter 다시 정의해서 청킹 사이즈 정하기

In [6]:
# 1. 임베딩 모델 다시 로드
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}
)

# 2. 기존 벡터 저장소 불러오기 (재임베딩 불필요)
from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name='tricare_rag',
    embedding_function=embedding_model,
    persist_directory='./chroma_db'
)

# 3. text_splitter 다시 정의
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_16228\2987918243.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
from tricare_exclusions_crawler import crawl_exclusions, to_documents

results = crawl_exclusions()
exclusion_docs = to_documents(results)

✅ 제외 항목 링크 55개 수집 완료
  [1/55] Acupuncture 수집 중...
  [2/55] Alterations to Living Space 수집 중...
  [3/55] Alternative Treatments 수집 중...
  [4/55] Assisted Living Facility Care 수집 중...
  [5/55] Augmentation Mammoplasty 수집 중...
  [6/55] Autopsy Services 수집 중...
  [7/55] Aversion Therapy 수집 중...
  [8/55] Camps 수집 중...
  [9/55] Charges for Missed Appointments 수집 중...
  [10/55] Computerized Dynamic Posturography (CDP) 수집 중...
  [11/55] Cosmetic Drugs 수집 중...
  [12/55] Domiciliary Care 수집 중...
  [13/55] Dry Needling 수집 중...
  [14/55] Dynamic Posturography 수집 중...
  [15/55] Dyslexia Treatment 수집 중...
  [16/55] Elective Psychotherapy and Mind Expansion Psychotherapy 수집 중...
  [17/55] Elective Services or Supplies 수집 중...
  [18/55] Elevators or Chair Lifts 수집 중...
  [19/55] Exercise Equipment 수집 중...
  [20/55] Exercise Programs 수집 중...
  [21/55] Experimental Procedures 수집 중...
  [22/55] Fluoride Preparations 수집 중...
  [23/55] Gym Membership 수집 중...
  [24/55] Homeopathic and Herbal Drugs 수집 중...
 

In [8]:
# 기존 text_splitter 재사용
exclusion_chunks = text_splitter.split_documents(exclusion_docs)

print(f'🔪 제외 항목 청크 수: {len(exclusion_chunks)}개')

🔪 제외 항목 청크 수: 64개


In [9]:
# 기존 vector_store에 추가
vector_store.add_documents(exclusion_chunks)
print(f'✅ 추가 후 벡터 수: {vector_store._collection.count()}개')

✅ 추가 후 벡터 수: 128개


---

# 기존 DEMO 1,2,3 모덷 로드하기
1. 환경 변수 로드
2. 임베딩 모델 로드(벡터 저장소 불러올 때 필요함)
3. 기존 벡터 저장소 불러오기
4. retire 재정의하기 
5. LLM 재정의하기
6. 프롬프트 재조립
=> 결과 확인

In [10]:
# 환경 변수 로드
import os
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
# 임베딩 모델 로드
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [12]:
# 기존 벡터 저장소 불러오기
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name='tricare_rag',
    embedding_function=embedding_model,
    persist_directory='./chroma_db'
)
table_vector_store = Chroma(
    collection_name='tricare_cost_tables',
    embedding_function=embedding_model,
    persist_directory='./chroma_db2'
)

In [13]:
# retriever 재정의
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 6}
)
table_retriever = table_vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [14]:
# LLM 재정의
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

In [ ]:
#!pip install langchain-teddynote

In [ ]:
# 프롬프트 + 체인 재조립
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import  RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
PROMPT_TEMPLATE = ('안녕하세요. 사용자님 \
                   궁금한 내용: {question} 접수 받았습니다. 곧 답변 드리겠습니다.\
                    🗂️ 파일 정리 중..')
rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

def format_docs(docs):
    ...

rag_chain = (
    {'question': RunnablePassthrough(), 'context': retriever | format_docs}
    | rag_prompt | model | StrOutputParser()
)
table_rag_chain = (
    {'question': RunnablePassthrough(), 'context': table_retriever | format_docs}
    | rag_prompt | model | StrOutputParser()
)

In [17]:
print('✅ 준비 완료')

✅ 준비 완료


---

# DEMO 1,2,3 사용

In [33]:
# 질문 설정
question = '보험 가입 후 취소하고 싶을 때 전액 환불이 가능한 청약 철회 기간(Free Look Period)은 며칠인가요?'
# question = 'I plan on still working when I turn 65. According to Medicare, I can wait to sign up for Medicare Part B since I’ll continue to have health insurance through my employer. How does this affect my TRICARE?'
#question = 'What is the TRICARE Prime Demo?'


In [34]:
# Demo 2 — RAG (텍스트 + CSV + 제외 항목)
print('=' * 65)
print('📚 [Demo 2] RAG 답변 — 텍스트/CSV/제외항목 문서 검색')
print('=' * 65)

retrieved = retriever.invoke(question)
print('🔎 검색된 청크:')
for i, chunk in enumerate(retrieved):
    src = chunk.metadata.get('source', '')
    print(f'  [{i+1}] {src}: {chunk.page_content[:80]}...')

print()
print(rag_chain.invoke(question))

📚 [Demo 2] RAG 답변 — 텍스트/CSV/제외항목 문서 검색
🔎 검색된 청크:
  [1] https://www.tricare.mil/CoveredServices/IsItCovered/AlterationsLivingSpace: 내용: TRICARE doesn’t cover alterations to living space. While TRICARE doesn’t cov...
  [2] https://www.tricare.mil/CoveredServices/IsItCovered/AlterationsLivingSpace: 내용: TRICARE doesn’t cover alterations to living space. While TRICARE doesn’t cov...
  [3] https://www.tricare.mil/CoveredServices/IsItCovered/LongTermCare: 내용: TRICARE doesn’t cover long term care. You can discuss exceptions or partial ...
  [4] https://www.tricare.mil/CoveredServices/IsItCovered/LongTermCare: 내용: TRICARE doesn’t cover long term care. You can discuss exceptions or partial ...
  [5] https://www.tricare.mil/CoveredServices/IsItCovered/LASIKSurgery: 제외 항목: LASIK Surgery
내용: TRICARE doesn’t cover LASIK eye surgery. This list of c...
  [6] https://www.tricare.mil/CoveredServices/IsItCovered/LASIKSurgery: 제외 항목: LASIK Surgery
내용: TRICARE doesn’t cover LASIK eye surgery. This list of 

**사용 방법 요약**
```
질문 바꾸고 싶을 때  → question 변수만 수정
Demo 1만 쓰고 싶을 때 → Demo 1 셀만 실행
Demo 2, 3 비교하고 싶을 때 → 두 셀 순서대로 실행
```

In [30]:
# Demo 3 — RAG (표 전용)
print('=' * 65)
print('📊 [Demo 3] RAG 답변 — 비용 표 전용 검색')
print('=' * 65)
print(table_rag_chain.invoke(question))


📊 [Demo 3] RAG 답변 — 비용 표 전용 검색
안녕하세요! TRICARE Prime Demo는 미국 국방부의 TRICARE 프로그램의 일환으로, 특정 지역에서 제공되는 의료 서비스 모델입니다. 이 프로그램은 군인과 그 가족들에게 보다 나은 의료 서비스를 제공하기 위해 설계되었습니다. TRICARE Prime Demo는 일반적으로 의료 서비스 접근성을 향상시키고, 비용을 절감하며, 예방적 치료를 촉진하는 데 중점을 둡니다.

이 프로그램은 특정 지역에서 시범적으로 운영되며, 참여자들은 지정된 의료 제공자와 협력하여 치료를 받게 됩니다. TRICARE Prime Demo의 주요 목표는 군인과 그 가족들이 필요한 의료 서비스를 보다 쉽게 이용할 수 있도록 하는 것입니다.

더 궁금한 점이 있으시면 언제든지 질문해 주세요!
